In [1]:
import json
import pandas as pd
import gc
import datetime
from typing import Dict, TypedDict
from langchain_huggingface import ChatHuggingFace, HuggingFacePipeline
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
import torch
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceBgeEmbeddings
from langgraph.graph import StateGraph, END
import warnings

warnings.filterwarnings("ignore")


# 1. State Definition

class AgentState(TypedDict):
    goal: str
    plan: str
    context: str
    computed_sentiment: dict
    db_metadata: dict
    draft_report: str
    validation_errors: str
    retry_count: int


# 1.5 Global LLM Initialization

print("Loading Llama 3.1 8B into VRAM (This takes a moment)")
local_model_path = "/home/jovyan/models/meta-llama/Llama-3.1-8B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(local_model_path)
model = AutoModelForCausalLM.from_pretrained(local_model_path, torch_dtype=torch.bfloat16)

pipe = pipeline(
    "text-generation", 
    model=model, 
    tokenizer=tokenizer, 
    max_new_tokens=4096, 
    return_full_text=False, 
    do_sample=False,
    pad_token_id=tokenizer.eos_token_id,
    eos_token_id=tokenizer.eos_token_id,
    device=0
)
base_llm = HuggingFacePipeline(pipeline=pipe)
global_llm = ChatHuggingFace(llm=base_llm)


# 2. Node Functions

def plan_node(state: AgentState):
    print("\n--- [NODE] PLANNING ---")
    torch.cuda.empty_cache()
    gc.collect()
    
    prompt = f"Goal: {state['goal']}\nWrite a 3-step search plan to retrieve relevant business intelligence from a vector database. Output ONLY the numbered steps."
    plan = global_llm.invoke(prompt).content
    print(f"Agent Plan:\n{plan}")
    return {"plan": plan, "retry_count": 0, "validation_errors": ""}

def retrieve_node(state: AgentState):
    print("\n--- [NODE] RETRIEVAL & METADATA AGGREGATION ---")
    torch.cuda.empty_cache()
    gc.collect()
    
    embedder = HuggingFaceBgeEmbeddings(
        model_name="BAAI/bge-base-en-v1.5",
        model_kwargs={'device': 'cuda'},
        encode_kwargs={'normalize_embeddings': True}
    )
    db = Chroma(persist_directory="./chroma_db", embedding_function=embedder)
    retriever = db.as_retriever(search_kwargs={"k": 15}) 
    
    search_query = f"{state['goal']} {state['plan']}"
    docs = retriever.invoke(search_query)

    
    context_blocks = []
    for doc in docs:
        url = doc.metadata.get('url', 'N/A')
        date = doc.metadata.get('date', 'Unknown Date')
    
        # Injecting the date directly into the text the LLM reads
        context_blocks.append(f"Date Published: {date}\nText: {doc.page_content}\nSource URL: {url}")
    
    context_str = "\n\n".join(context_blocks)
    
    all_data = db.get(include=["metadatas"])
    df = pd.DataFrame(all_data["metadatas"])
    
    total_docs = len(df) if not df.empty else 0
    unique_sources = df['source'].nunique() if not df.empty else 0
    last_update = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S UTC")
    
    db_metadata = {
        "company_name": "NVIDIA Corporation",
        "industry": "Semiconductors & AI Computing",
        "total_documents": total_docs,
        "data_sources_count": unique_sources,
        "last_update": last_update
    }
    
    if not df.empty:
        news_df = df[df["source"] != "HackerNews"]
        hn_df = df[df["source"] == "HackerNews"]
        
        news_score = int(news_df["sentiment_score"].mean()) if not news_df.empty else 50
        pub_score = int(hn_df["sentiment_score"].mean()) if not hn_df.empty else 50
        
        df['date'] = pd.to_datetime(df['date'], format='mixed', utc=True)
        cutoff_date = pd.Timestamp.utcnow() - pd.Timedelta(days=90)
        df = df[df['date'] >= cutoff_date]
        
        trend_df = df.set_index('date').resample('W')['sentiment_score'].mean().fillna(50).reset_index()
        trend_data = [{"date": row['date'].strftime("%Y-%m-%d"), "score": int(row['sentiment_score'])} for _, row in trend_df.iterrows()]
    else:
        news_score, pub_score, trend_data = 50, 50, []

    computed_sentiment = {
        "news_sentiment_score": news_score,
        "public_sentiment_score": pub_score,
        "historical_trend": trend_data
    }
    
    print(f"Retrieved {len(docs)} documents. Generated deterministic metadata.")
    return {"context": context_str, "computed_sentiment": computed_sentiment, "db_metadata": db_metadata}

def generate_node(state: AgentState):
    print(f"\n--- [NODE] GENERATION (Attempt {state['retry_count'] + 1}) ---")
    torch.cuda.empty_cache()
    gc.collect()
    
    schema = """
    {
      "market_intelligence": {
        "recent_news": ["Bullet point 1", "Bullet point 2"],
        "competitor_activities": ["Bullet point 1"],
        "emerging_technologies": ["Bullet point 1"],
        "important_announcements": ["Bullet point 1"]
      },
      "opportunities": [
        {"title": "", "impact_level": "High/Medium/Low", "verbatim_quote_from_context": "", "confidence_score": 0}
      ],
      "risks": [
        {"title": "", "category": "", "severity_level": "High/Medium/Low", "verbatim_quote_from_context": "", "confidence_score": 0}
      ],
      "sentiment_analysis": {
        "news_sentiment_score": "Inject EXACT news_score here",
        "public_sentiment_score": "Inject EXACT pub_score here",
        "sentiment_justification": "One sentence analyzing why the market feels this way",
        "historical_trend": "Inject EXACT trend array here"
      },
      "recommendations": [
        {
          "recommendation": "", 
          "priority": "High/Medium/Low", 
          "supporting_evidence": "MUST BE A VERBATIM QUOTE", 
          "expected_impact": "", 
          "risk_level": "High/Medium/Low",
          "source_url": "URL MUST COME FROM CONTEXT"
        }
      ],
      "ceo_briefing": {
        "what_happened": "Summary of current events",
        "why_it_matters": "Strategic implication",
        "management_next_steps": "Actionable directive"
      }
    }
    """
    
    system_prompt = f"""You are an autonomous AI executive advisor.
    You MUST output valid JSON strictly matching this schema: {schema}
    
    Use the Pre-Computed Sentiment Math:
    {json.dumps(state['computed_sentiment'])}
    
    Use this Retrieved Context to ground your evidence and URLs:
    {state['context']}
    
    Previous Validation Errors to Fix (if any): {state['validation_errors']}

    CRITICAL INSTRUCTION FOR RECENT NEWS: 
    You are receiving text chunks that contain a "Date Published". 
    You MUST analyze the date and the context. 
    Do not list historical events (like the 2020 Arm acquisition attempt) in the "recent_news" array unless the text is specifically reporting a brand new development regarding it.

    CRITICAL COMPETITOR LOGIC: 
    When extracting hardware or product comparisons, you MUST verify the chronological subject. 
    If a new NVIDIA product (like RTX Spark) is being compared to an older competitor product (like Apple M2), do NOT list the older product as a new competitor activity. 
    Only extract actions taken by competitors in the last 90 days.
    """
    
    draft = global_llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=state['goal'])]).content
    return {"draft_report": draft}

def validate_node(state: AgentState):
    print("\n [NODE] VALIDATION ")
    torch.cuda.empty_cache()
    gc.collect()
    
    try:
        parsed_json = json.loads(state['draft_report'])
    except json.JSONDecodeError as e:
        print(f"Validation Failed: Invalid JSON Syntax -> {e}")
        return {"validation_errors": f"Your output was not valid JSON. Error: {e}. You MUST output pure JSON.", "retry_count": state['retry_count'] + 1}
    
    errors = []
    recommendations = parsed_json.get("recommendations", [])
    
    if not recommendations:
        errors.append("You failed to provide any recommendations in the JSON array.")
        
    for i, rec in enumerate(recommendations):
        source_url = rec.get("source_url", "")
        if source_url not in state["context"]:
            errors.append(f"Recommendation {i+1} hallucinated this URL: '{source_url}'. You MUST use an exact URL from the context.")
            
    if not errors:
        print("Validation Passed: No URL hallucinations detected by deterministic check.")
        parsed_json["company_overview"] = state["db_metadata"]
        return {"validation_errors": "VALID", "draft_report": json.dumps(parsed_json, indent=2)}
    else:
        error_msg = " ".join(errors)
        print(f"Validation Failed: {error_msg}")
        return {"validation_errors": error_msg, "retry_count": state['retry_count'] + 1}

def validation_router(state: AgentState):
    if state["validation_errors"] == "VALID" or state["retry_count"] >= 3:
        return "end"
    return "retry"


# 3. Graph Compilation

def build_agent():
    workflow = StateGraph(AgentState)
    workflow.add_node("planner", plan_node)
    workflow.add_node("retriever", retrieve_node)
    workflow.add_node("generator", generate_node)
    workflow.add_node("validator", validate_node)
    
    workflow.set_entry_point("planner")
    workflow.add_edge("planner", "retriever")
    workflow.add_edge("retriever", "generator")
    workflow.add_edge("generator", "validator")
    
    workflow.add_conditional_edges("validator", validation_router, {"end": END, "retry": "generator"})
    return workflow.compile()

if __name__ == "__main__":
    test_goal = "Analyze the current strategic threats to NVIDIA's supply chain, outline competitor activities, and identify new market opportunities based on developer sentiment."
    agent_app = build_agent()
    print("Initiating Agentic State Machine")
    final_state = agent_app.invoke({"goal": test_goal, "retry_count": 0})
    
    with open("ceo_intelligence_report.json", "w", encoding="utf-8") as f:
        f.write(final_state["draft_report"])
    print("\nMission Accomplished. JSON Artifact saved.")

/home/jovyan/vault/AI_CEO_NLP/ai_ceo_env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/tmp/ipykernel_172/2065191125.py:10: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import Chroma


Loading Llama 3.1 8B into VRAM (This takes a moment)


Loading weights: 100%|██████████| 291/291 [00:00<00:00, 6186.41it/s]


Initiating Agentic State Machine

--- [NODE] PLANNING ---
Agent Plan:
1. **Step 1: Define Search Query and Vector Space**
   - Identify relevant keywords and phrases related to NVIDIA's supply chain, competitors, and market opportunities, such as "NVIDIA supply chain disruption," "GPU market competition," "developer sentiment analysis," and "artificial intelligence market trends."
   - Create a vector space model that captures the semantic relationships between these keywords and phrases to enable efficient retrieval of relevant information.

2. **Step 2: Retrieve and Preprocess Data**
   - Utilize the vector space model to query the vector database and retrieve a large dataset of relevant documents, articles, and social media posts related to NVIDIA's supply chain, competitors, and market opportunities.
   - Preprocess the retrieved data by removing stop words, stemming or lemmatizing words, and converting all text to lowercase to improve the accuracy of the analysis.

3. **Step 3: An

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5923.99it/s]


Retrieved 15 documents. Generated deterministic metadata.

--- [NODE] GENERATION (Attempt 1) ---

 [NODE] VALIDATION 
Validation Passed: No URL hallucinations detected by deterministic check.

Mission Accomplished. JSON Artifact saved.
